### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import json

DATA_ROOT = next((p for p in (Path("../../data"), Path("data"), Path("downloads/python/data")) if p.exists()), Path("data"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_URLS = {
    "districts.geojson": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp09/districts.geojson",
    "services.csv": "https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp09/services.csv"
}

def valid_cached(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if path.suffix.lower() == ".gpkg":
            return path.read_bytes()[:16] == b"SQLite format 3\x00"
        if path.suffix.lower() == ".geojson":
            json.loads(path.read_text(encoding="utf-8"))
        return True
    except Exception:
        return False

for filename, url in DATA_URLS.items():
    target = DATA_ROOT / "gp09" / filename
    if valid_cached(target):
        print(f"[OK] Ήδη διαθέσιμο: {target}")
        continue
    target.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(url, timeout=30) as response:
            payload = response.read()
        if not payload:
            raise RuntimeError("κενή απάντηση")
        target.write_bytes(payload)
        print(f"[OK] Κατέβηκε: {target}")
    except Exception as exc:
        if target.exists():
            target.unlink()
        print(f"[WARN] Δεν ήταν δυνατή η λήψη {url}: {exc}")


# GP09 — Spatial Join

Σύνδεσε σημεία υπηρεσιών με τη διοικητική περιοχή όπου βρίσκονται. Πριν από τη syntax αποφάσισε: ποιο dataset είναι left, ποιο right, ποιο predicate και αν θέλεις `left` ή `inner` join. Το spatial join μεταφέρει attributes· δεν κόβει geometry.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
root = DATA_ROOT / "gp09"
df = pd.read_csv(root / 'services.csv')
services = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs='EPSG:4326')
districts = gpd.read_file(root / 'districts.geojson')


Τα σημεία είναι το left dataset και οι περιοχές το right dataset. Χρησιμοποίησε predicate `within`: κάθε point πρέπει να βρίσκεται μέσα σε polygon. Το `left` κρατά και unmatched points, άρα είναι χρήσιμο για QA· το `inner` κρατά μόνο matches.

In [ ]:
joined = gpd.sjoin(services, districts, how='left', predicate='within')
joined.head()


In [ ]:
counts = joined.dropna(subset=['district_id']).groupby('district_name').size()
print(counts)


QA: σύγκρινε counts πριν/μετά, unmatched rows, CRS και geometry retention. Εξήγησε αν το αποτέλεσμα είναι one-to-one ή one-to-many.

In [ ]:
assert len(joined) >= len(services)
assert joined.geometry.geom_type.eq('Point').all()
assert joined.crs.to_epsg() == 4326
print('unmatched:', joined['district_id'].isna().sum())


In [ ]:
joined.to_file('gp09_services_by_district.gpkg', layer='services', driver='GPKG')
counts.to_csv('gp09_service_counts.csv')
print('export complete')
